Pauli Matrix Decomposition

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.linalg as LA

# Qiskit 2.x: import explicit, bukan wildcard
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit.quantum_info import Operator, SparsePauliOp, Statevector

In [ ]:
# ============================================================
# Script 4.2 (Bug-fixed): Hamiltonian
# ============================================================

J  = 1
h0 = -0.5
h1 =  0.5

X = np.array([[0, 1 ], [1,  0]], dtype=complex)
Y = np.array([[0,-1j], [1j, 0]], dtype=complex)  # FIX: tanda -1j di [0,1]
Z = np.array([[1, 0 ], [0, -1]], dtype=complex)
I = np.eye(2,             dtype=complex)

# Hamiltonian 4×4 untuk sistem 2 qubit
# np.kron = Kronecker product (tensor product ⊗)
# Analogi: np.kron(Z, I) artinya "Z bekerja di qubit-0, I (identitas) di qubit-1"
H = (0.5 * (h0 * np.kron(Z, I) + h1 * np.kron(I, Z))
     + (J / 4) * (np.kron(X, X) + np.kron(Y, Y) + np.kron(Z, Z)))


print("Hamiltonian H (4×4):")
print(np.round(H, 3))


Hamiltonian H (4×4):
[[ 0.25+0.j  0.  +0.j  0.  +0.j  0.  +0.j]
 [ 0.  +0.j -0.75+0.j  0.5 +0.j  0.  +0.j]
 [ 0.  +0.j  0.5 +0.j  0.25+0.j  0.  +0.j]
 [ 0.  +0.j  0.  +0.j  0.  +0.j  0.25+0.j]]


In [14]:
# ============================================================
# Script 4.11 (Bug-fixed): Utility Functions
# ============================================================

def vec_query(arr, my_dict):
    """
    Vectorized dictionary lookup.
    
    Analogi: daripada pergi ke perpustakaan satu per satu untuk tiap buku,
    kita kirim daftar belanja sekaligus dan dapat semua buku sekaligus.
    """
    return np.vectorize(my_dict.__getitem__, otypes=[object])(arr)


def nested_kronecker_product(a):
    """
    Recursive Kronecker product: kron(a[0], kron(a[1], kron(a[2], ...)))
    
    Analogi: seperti menggabungkan "layer" lego satu per satu dari kanan ke kiri.
    nested_kronecker_product([Z, X, Y]) = Z ⊗ X ⊗ Y
    """
    if len(a) == 2:
        return np.kron(a[0], a[1])
    else:
        return np.kron(a[0], nested_kronecker_product(a[1:]))


def Hilbert_Schmidt(mat1, mat2):
    """
    Hilbert-Schmidt inner product: Tr(mat1† · mat2)
    
    Analogi: seperti dot product antara dua vektor, tapi untuk matriks.
    Mengukur "seberapa mirip" mat1 dengan mat2 dalam ruang matriks.
    
    BUG FIX: paper menggunakan '*' (element-wise), seharusnya '@' (matmul)
    """
    return np.trace(mat1.conj().T @ mat2)  # FIX: @ bukan *

In [15]:
# ============================================================
# Script 4.12 (Bug-fixed): Pauli Decomposition Function
# ============================================================
import numpy as np
import itertools

def decompose(Ham_arr, tol=10):
    pms = {
        'I': np.eye(2, dtype=complex),
        'X': np.array([[0, 1 ], [1,  0]], dtype=complex),
        'Y': np.array([[0,-1j], [1j, 0]], dtype=complex),
        'Z': np.array([[1, 0 ], [0, -1]], dtype=complex),
    }
    pauli_keys = list(pms.keys())
    nqb = int(np.log2(Ham_arr.shape[0]))
    result_dict = {}

    sigma_combinations = list(itertools.product(pauli_keys, repeat=nqb))
    for combo in sigma_combinations:
        pauli_str = ''.join(combo)
        mat_list  = vec_query(np.array(combo), pms)
        p_matrix  = nested_kronecker_product(mat_list)
        norm_factor = 1.0 / (2**nqb)
        a_coeff = norm_factor * Hilbert_Schmidt(p_matrix, Ham_arr)
        if abs(a_coeff) > 10**(-tol):
            result_dict[pauli_str] = np.round(a_coeff.real, tol)
    return result_dict

# ✅ FIX: was decompose(Ham_arr) — Ham_arr was never defined, correct variable is H
decomp = decompose(H)
print("Dekomposisi Pauli dari Hamiltonian 2-site Heisenberg:")
for term, coeff in decomp.items():
    print(f"  {coeff:+.6f} * {term}")

# Rekonstruksi dan verifikasi
pms_full = {
    'I': np.eye(2, dtype=complex),
    'X': np.array([[0,1],[1,0]], dtype=complex),
    'Y': np.array([[0,-1j],[1j,0]], dtype=complex),
    'Z': np.array([[1,0],[0,-1]], dtype=complex),
}
H_reconstructed = np.zeros((4,4), dtype=complex)
for term, coeff in decomp.items():
    mats = [pms_full[c] for c in term]
    H_reconstructed += coeff * nested_kronecker_product(mats)

print(f"\nRekonstruksi benar: {np.allclose(H, H_reconstructed, atol=1e-8)} ✓")

Dekomposisi Pauli dari Hamiltonian 2-site Heisenberg:
  +0.250000 * IZ
  +0.250000 * XX
  +0.250000 * YY
  -0.250000 * ZI
  +0.250000 * ZZ

Rekonstruksi benar: True ✓
